# Agenda

1. Locals vs. globals
2. Modules and Python's standard library

# MIT's AI and education report

- https://aiandeducation.mit.edu/
- https://lernerpython.com/2026/08/26/mit-just-called-for-an-educational-revolution/

# Local vs. global variables

When we're working without any functions, all of our variables are *global*. That means:

- Any part of our program can assign to a variable
- Any part of our program can modify a value via a variable
- Any part of our program can read from a variable

This means that if one part of your program assigns to `x`, and another part of your program reads from `x`, that's perfectly fine and acceptable.

When we introduce functions, things get more complex. That's because variables within a function are considered *local*. That is, they are private to the function, and only exist so long as the function is running. If you define a variable inside of a function, then that variable will not be around when the function exits.

Why would they do this? So that you can have different functions with the same variable name. In most programming languages, the variables that you have inside of a function are local, and thus private, and so you don't need to worry about "namespace collisions."

However, this does raise all sorts of questions:

- If a variable in a function is local, how can it read data from the outside (i.e., globals)?
- How can it assign to global variables?
- How can it modify values via global variables?

# A basic rule for local variables

If you're inside of a function, and you assign to a variable, then that variable is now local. The variable will disappear when the function returns. This means that any assignment to any variable inside of a function instantly creates a local variable.

BTW, parameters are always local variables, too.

In [1]:
def add(x, y):
    return x + y

In [2]:
add(10, 3)

13

In [3]:
x

NameError: name 'x' is not defined

In [4]:
y

NameError: name 'y' is not defined

In [5]:
def add(x, y):
    total = x + y
    return total

In [6]:
add(12, 14)

26

In [7]:
x

NameError: name 'x' is not defined

In [8]:
y

NameError: name 'y' is not defined

In [9]:
total

NameError: name 'total' is not defined

Not having "leakage" of local variables to the global scope makes it easier to write functions and to work with them. I don't have to worry that if I call a function, some variable is suddenly, surprisingly, going to change its value. It also means that someone who writes a function doesn't need to worry about their variables colliding with someone else's variables.

Does this mean that a function cannot modify a global value? It can! A variable can be local or global, but all values are theoretically available at all times, so long as we have access to them via a variable.

If I'm in a function, and I'm using a local variable that refers to a value that a global *also* refers to, then I can (from within the function) modify that value, mutate that data structure, and thus affect something that we might imagine is global.

In [10]:
mylist = [10, 20, 30]   # global variable mylist

def myfunc(x):          # function myfunc, with a parameter (i.e., local variable) x
    x.append(40)        # append to the list that x refers to, which is the same as mylist!

print(mylist)           # print the global
myfunc(mylist)          # invoke myfunc, assigning (global) mylist to (local) x -- at which point they both refer to the same value
print(mylist)

[10, 20, 30]
[10, 20, 30, 40]


If you modify a mutable value via a local variable, and if there's a global that also refers to it, then you will indeed modify the global.

That's not a bad thing, but you do want to document it -- probably in your docstring.

This is very different from assigning to a variable.

```python
x = 100     # this is assigning to a variable -- this should *not* be done in a function, and gets confusing
x[1] = 100  # this is modify a value that already exists -- this happens all of the time in a function
```

If you want to keep track of things from within your function, and the function might be invoked numerous times, then one option is for you to have an agreed-upon global variable that each invocation of the function modifies.

For example, you might have a global variable that refers to a dict. Each time you invoke a function, it modifies/updates/mutates the dictionary.

Bottom line: If there is a global variable that refers to a list or dict, you can definitely update/modify it from within a function. Python will figure out that you're working with a global, and assign/update accordingly.

However, the moment that you just *assign* to a variable in a function, you're creating a local variable, and in the worst case, it'll "shadow* the global, meaning that the global won't be visible because you created a high-priority local.

# Exercise: Updating a global

1. Write a function, `count_characters`, that takes a string.
2. The function should iterate over the characters in a string:
    - If we have seen this character before, then add 1 to the existing value in a global dict, `counts`, where characters are keys and values are values.
    - If we have never seen this character before, then add a key-value pair for the character and the count being 1.
4. The `counts` dict is global, not defined inside of the function.
5. Repeatedly ask the user for input:
    - If they give an empty string, stop
    - If they give a regular string, go through each character and add as in step 2.

https://practice.lernerpython.com/classroom/ce06963a8d/ex-39

In [14]:
counts = {}    # this is a global variable -- defined outside of the function

def count_characters(text):
    for one_character in text:
        if one_character in counts:     # if one_character is a key in counts
            counts[one_character] += 1  # ... add 1 to the value in the dict for one_character
        else:
            counts[one_character] = 1   # if not, then add the new key-value pair

while True:
    user_string = input('Enter text: ').strip()

    if user_string == '':
        break

    count_characters(user_string)

print(counts)

KeyboardInterrupt: Interrupted by user

In [13]:
counts

{'h': 2, 'e': 3, 'l': 2, 'o': 2, ' ': 2, 'u': 1, 't': 2, 'r': 1}

# What should we think?

- On the one hand, it's very convenient that we can update a global dict/list from within our function
- On the other hand, we're asking for trouble, aren't we? What if more than one function will update this variable?

And so, another technique is (I'd say) more popular:

- Pass the value to the function as an argument
- Have the function modify that argument
- Have the function return the argument (optional)

This takes us out of the business of talking directly to global variables from within our functions. Rather, we'll just deal with the arguments we get, and thus local variables.

# Exercise: Pass the dict as an argument

Redo the previous exercise (it's OK to start with my code on GitHub), but change it:

- The function takes two arguments -- a string and a `counts` global
- The function will modify the `counts` global that it gets
- It should also return that modified global

https://practice.lernerpython.com/classroom/ce06963a8d/ex-41

In [ ]:
# it's a good idea to pass arguments, even if they represent global values, into a function.
# it just makes it easier to track who is doing what and when.

counts = {}   

def count_characters(text, count_dict):
    for one_character in text:
        if one_character in count_dict:     # if one_character is a key in counts
            count_dict[one_character] += 1  # ... add 1 to the value in the dict for one_character
        else:
            count_dict[one_character] = 1   # if not, then add the new key-value pair
    return count_dict   # when we're done, return this dict

while True:
    user_string = input('Enter text: ').strip()

    if user_string == '':
        break

    count_characters(user_string, counts)

print(counts)

When you mention a variable in a function, Python looks for that variable in a set order (LEGB -- local, enclosing, global, and builtin). We're just talking about the two most common levels, *L*ocal and *G*lobal.

If your function mentions a variable that isn't defined in the function (no assignment to it, and no parameter by that name), Python then looks for a global of that name. If it finds it, it'll read from it or mutate it.

# Assignment ≠ mutation

If you assign to a variable in a Python function, then that variable is local. Period. 

You can accidentally create a local variable by thinking that you're assigning to a global. But any assignment in a function creates (or updates) a local, unless you've told Python otherwise.

If there is a global integer `x`, and you (inside of your function) say `x = 100`, then you have not modified the global `x`. Rather, you have created a local variable `x`. Now, because of the LEGB rule, you will only (easily) be able to retrieve the local `x`, and you won't be able to easily retrieve the global `x`. This is known as "shadowing." 

# Bottom line

1. Don't assign to variables unless you want them to be local. If you have a global variable you want to assign to from within a function... find another way.
2. Mutation of a global, by contrast, works just fine -- either by assignment or using a method (e.g., `list.append`). However, you should think about whether you want to be modifying globals in this way.
3. A nice way to ensure that you aren't accidentally changing globals you shouldn't would be to pass an argument, and then work with that. 